# 03 Lasso 回归

Lasso 是带 L1 正则化的线性回归。它最重要的特点是：可以把一部分系数压到精确的 0，因此常被用来做特征选择。


## 0. 学习目标和阅读地图

Lasso 的重点是“稀疏”：它不仅让模型变简单，还能让某些特征的系数变成 0。

你需要掌握：

1. L1 正则为什么会产生稀疏解。
2. Lasso 和 Ridge 在特征选择上的差异。
3. `alpha` 如何控制保留多少特征。
4. 为什么相关特征会让 Lasso 的解释变得不稳定。


## 1. 数学逻辑

Lasso 的目标函数是：

$$L(w)=\frac{1}{2n}\sum_{i=1}^{n}(y_i-X_iw)^2 + \alpha\sum_{j=1}^{d}|w_j|$$

L1 正则项使用绝对值：

$$||w||_1 = \sum_j |w_j|$$

和 Ridge 的 L2 不同，L1 的几何形状更容易让最优解落在坐标轴上，所以一些系数会变成 0。


## 1.1 推导拆开看：L1 为什么会把系数压成 0

Lasso 惩罚的是绝对值：

$$||w||_1 = |w_1| + |w_2| + \cdots + |w_d|$$

绝对值函数在 0 点有一个尖角。优化时，这个尖角会让很多参数停在 0。

一维 soft-thresholding 可以体现这个效果：

$$S(z, \alpha)=\begin{cases}z-\alpha, & z>\alpha\\0, & |z|\le \alpha\\z+\alpha, & z<-\alpha\end{cases}$$

如果一个特征对目标的贡献不够强，它对应的 `z` 会落在 `[-alpha, alpha]` 之间，系数直接变成 0。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

n, d = 150, 12
X = np.random.normal(size=(n, d))
true_w = np.array([4.0, -3.0, 2.0] + [0.0] * (d - 3))
y = X @ true_w + np.random.normal(scale=1.0, size=n)

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)


## 1.2 这个例子的设计目的

数据里有 12 个特征，但真实有用的只有前 3 个。Lasso 的目标不是只把 MSE 做低，而是希望自动发现“哪些特征值得保留”。

这类设置很常见：真实业务里我们经常有很多候选特征，但并不知道哪些真正有用。


In [ ]:
# 从零实现：一维 soft-thresholding 是 Lasso 的核心直觉
# 如果普通梯度想把系数推得不够大，L1 会直接把它压成 0。

def soft_threshold(z, gamma):
    if z > gamma:
        return z - gamma
    if z < -gamma:
        return z + gamma
    return 0.0

for z in [-2, -0.5, 0.2, 1.5, 3.0]:
    print(f'z={z: .1f} -> soft_threshold(z, 1.0)={soft_threshold(z, 1.0): .1f}')

# 一个简化版坐标下降：逐个更新每个系数
alpha = 0.08
w = np.zeros(d)
b = y_train.mean()
Xc = X_train_s
yc = y_train - b

for epoch in range(80):
    for j in range(d):
        residual = yc - Xc @ w + Xc[:, j] * w[j]
        rho = np.mean(Xc[:, j] * residual)
        w[j] = soft_threshold(rho, alpha) / (np.mean(Xc[:, j] ** 2) + 1e-12)

print('真实系数:', true_w)
print('从零 Lasso 系数:', np.round(w, 3))


## 1.3 从零实现代码怎么读

这段从零代码展示的是坐标下降 coordinate descent：

1. 固定其他权重，只更新第 `j` 个权重。
2. 计算当前特征和残差的相关性 `rho`。
3. 对 `rho` 做 soft-thresholding。
4. 重复多轮，直到权重稳定。

它不是最快实现，但很适合理解 Lasso 为什么会做特征选择。


In [ ]:
# sklearn 实战
ols = LinearRegression().fit(X_train_s, y_train)
lasso = Lasso(alpha=0.08, max_iter=10000).fit(X_train_s, y_train)

for name, model in [('LinearRegression', ols), ('Lasso', lasso)]:
    pred = model.predict(X_test_s)
    print(name)
    print('  weights:', np.round(model.coef_, 3))
    print('  非零系数数量:', np.sum(np.abs(model.coef_) > 1e-8))
    print('  MSE:', round(mean_squared_error(y_test, pred), 3))

alphas = np.logspace(-3, 0, 50)
coefs = np.array([Lasso(alpha=a, max_iter=10000).fit(X_train_s, y_train).coef_ for a in alphas])
plt.plot(alphas, coefs)
plt.xscale('log')
plt.title('alpha 越大，Lasso 会把更多系数压到 0')
plt.xlabel('alpha')
plt.ylabel('coefficient')
plt.show()


In [ ]:
# 诊断：alpha 如何影响“保留的特征数量”和测试误差
alphas = np.logspace(-3, 0, 30)
nonzero_counts = []
test_mse = []
for a in alphas:
    m_lasso = Lasso(alpha=a, max_iter=10000).fit(X_train_s, y_train)
    nonzero_counts.append(np.sum(np.abs(m_lasso.coef_) > 1e-8))
    test_mse.append(mean_squared_error(y_test, m_lasso.predict(X_test_s)))

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(alphas, nonzero_counts, marker='o')
plt.xscale('log')
plt.title('alpha 与非零特征数量')
plt.xlabel('alpha')
plt.ylabel('non-zero coefficients')

plt.subplot(1, 2, 2)
plt.plot(alphas, test_mse, marker='o')
plt.xscale('log')
plt.title('alpha 与测试 MSE')
plt.xlabel('alpha')
plt.ylabel('MSE')
plt.tight_layout()
plt.show()


## 2.1 如何诊断 Lasso

Lasso 需要同时看预测效果和稀疏程度。如果测试误差只小幅变差，但特征数量大幅减少，模型可能更容易解释和部署。

但要注意：如果两个特征高度相关，Lasso 可能留下其中一个、丢掉另一个。这不代表被丢掉的特征在现实中无意义。


## 2. 常见误区

- Lasso 对特征尺度非常敏感，通常必须标准化。
- 当多个特征高度相关时，Lasso 可能随机留下其中一个，解释时要小心。
- Lasso 做的是线性特征选择，不代表被压成 0 的变量在真实世界中一定无意义。

## 3. 小实验

- 改 `alpha`，观察非零系数数量。
- 增加相关特征，看 Lasso 选择哪个。
- 对比 Ridge：Ridge 更平滑，Lasso 更稀疏。


## 5. 复习清单

- Lasso = 线性回归 + L1 正则。
- L1 能产生精确 0，因此可以做特征选择。
- 标准化几乎是必须的。
- 稀疏不等于真因果，只是模型选择结果。
